# Face Occlusion — Kaggle Training Notebook

## 1. Install & Clone

In [ ]:
import os

# Only upgrade lightweight deps — do NOT reinstall PyTorch.
# Kaggle's built-in PyTorch works on T4/P100 only when the right GPU is selected.
# --> Settings → Accelerator → GPU T4 x2  (P100 / sm_60 is incompatible with PyTorch 2.x)
!pip install -q --upgrade timm wandb tqdm

REPO_DIR = "/kaggle/working/Face-Occlusion-Prediction"
if os.path.isdir(REPO_DIR):
    os.chdir(REPO_DIR)
    !git pull
else:
    !git clone https://github.com/LeHoang510/Face-Occlusion-Prediction.git {REPO_DIR}
    os.chdir(REPO_DIR)

import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Working dir:", os.getcwd())

## 2. Prepare Dataset

In [ ]:
import os

KAGGLE_INPUT = "/kaggle/input/datasets/lhongnguyn/face-occlusion-data"

img_root  = f"{KAGGLE_INPUT}/crops/Crop_224_5fp_100K"
train_csv = f"{KAGGLE_INPUT}/DataChallenge2026/occlusion_datasets/train.csv"
test_csv  = f"{KAGGLE_INPUT}/DataChallenge2026/occlusion_datasets/test_students.csv"

assert os.path.isdir(img_root),   f"Not found: {img_root}"
assert os.path.isfile(train_csv), f"Not found: {train_csv}"
assert os.path.isfile(test_csv),  f"Not found: {test_csv}"

print("All paths OK.")
print("img_root :", img_root)
print("train_csv:", train_csv)
print("test_csv :", test_csv)

## 3. WandB Login

In [ ]:
from kaggle_secrets import UserSecretsClient
import os

os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
print("WandB API key loaded.")

## 4. Configure & Train

Edit `MODEL_NAME` below to pick the phase-2 balanced-batching model you want to train.

In [ ]:
from pathlib import Path
import yaml

# Pick one: "swin_t", "convnext_tiny", "vit_b_16"
MODEL_NAME = "swin_t"

CONFIG_BY_MODEL = {
    "swin_t": "src/data_challenge/configs/phase2_balanced_batching/swin_t.yaml",
    "convnext_tiny": "src/data_challenge/configs/phase2_balanced_batching/convnext_tiny.yaml",
    "vit_b_16": "src/data_challenge/configs/phase2_balanced_batching/vit_b_16.yaml",
}

assert MODEL_NAME in CONFIG_BY_MODEL, f"Unknown MODEL_NAME: {MODEL_NAME}"
SOURCE_CONFIG = CONFIG_BY_MODEL[MODEL_NAME]
assert Path(SOURCE_CONFIG).is_file(), f"Missing config: {SOURCE_CONFIG}. Push the phase-2 code before running on Kaggle."

CONFIG = f"/kaggle/working/{MODEL_NAME}_kaggle.yaml"

with open(SOURCE_CONFIG) as f:
    cfg = yaml.safe_load(f)

# Point to Kaggle input directly, no copying needed.
cfg["data"]["img_root"] = img_root
cfg["data"]["train_csv"] = train_csv
cfg["data"]["test_csv"] = test_csv
cfg["output"]["dir"] = "/kaggle/working/outputs"

Path(CONFIG).parent.mkdir(parents=True, exist_ok=True)
with open(CONFIG, "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print(f"Model    : {MODEL_NAME}")
print(f"Source   : {SOURCE_CONFIG}")
print(f"Config   : {CONFIG}")
print(f"Backbone : {cfg['model']['backbone']}")
print(f"Batching : {cfg['training'].get('batching', {}).get('strategy', 'random')}")
print(f"img_root : {cfg['data']['img_root']}")
print(f"Output   : {cfg['output']['dir']}")

In [ ]:
!PYTHONPATH=src python -m data_challenge.train --config {CONFIG}

## 5. Save Checkpoint to WandB Artifact

In [ ]:
import json
from pathlib import Path
import wandb

summary_paths = sorted(
    Path("/kaggle/working/outputs").glob("*/run_summary.json"),
    key=lambda path: path.stat().st_mtime,
)

if not summary_paths:
    print("No completed run found; training may have failed. Skipping artifact upload.")
else:
    summary_path = summary_paths[-1]
    run_dir = summary_path.parent
    best_ckpt = run_dir / "best_model.pt"
    config_path = run_dir / "config.yaml"
    assert best_ckpt.is_file(), f"Missing checkpoint: {best_ckpt}"
    assert config_path.is_file(), f"Missing copied config: {config_path}"

    with open(summary_path) as f:
        summary = json.load(f)

    wandb_kwargs = {"project": "data-challenge-occlusion"}
    if summary.get("wandb_run_id"):
        wandb_kwargs.update({"id": summary["wandb_run_id"], "resume": "must"})
    else:
        wandb_kwargs.update({"name": f"{summary['run_name']}_artifact"})

    run = wandb.init(**wandb_kwargs)
    artifact = wandb.Artifact(summary["run_name"], type="model", metadata=summary)
    artifact.add_file(str(best_ckpt), name="best_model.pt")
    artifact.add_file(str(summary_path), name="run_summary.json")
    artifact.add_file(str(config_path), name="config.yaml")
    run.log_artifact(artifact)
    run.finish()

    print(f"Artifact '{summary['run_name']}' saved to wandb.")
    print(f"Run dir: {run_dir}")
    print(f"Best score: {summary['best_val_score']:.6f} at epoch {summary['best_epoch']}")

## 6. Pull checkpoint back to local machine

Run this on your **local machine** after the Kaggle run finishes. Replace the artifact name with the `run_name` printed above, for example `phase2_balanced_swin_t:latest`.

```bash
.venv/bin/python -c "
import wandb
api = wandb.Api()
artifact = api.artifact('lehoang510/data-challenge-occlusion/phase2_balanced_swin_t:latest')
artifact.download('outputs/phase2_balanced_swin_t_kaggle/')
print('Done.')
"
```